# SELFIE Writer / Editor Probe

**Three-arm comparison of agent communication via hidden states**

| Arm | Key | What it does |
|-----|-----|--------------|
| A | `editor_verdict` | Editor receives the **raw essay text** |
| B | `editor_selfhs_verdict` | Editor's **own hidden states** over the draft span are re-injected at the embedding layer |
| C | `editor_writerhs_verdict` | The **writer's final hidden states** are injected into the editor at the draft placeholder positions |
| Self-probe | `writer_selfprobe_output` | Writer's final HS are injected back into the **writer** with a user-configurable prompt (default: 'repeat this message') — sanity check that the HS actually encode the essay content |

Comparing A, B, C isolates where communication loss occurs between agents:
- **A ≈ B** → editor's internal representation and surface text carry the same information  
- **A ≈ C** → writer's hidden states carry the same information as its decoded text  
- **B ≈ C** → writer and editor form compatible internal representations  
- **self-probe ≈ essay** → writer's HS genuinely encode the essay (if this fails, Arm C can't succeed)

### Qwen3 / Qwen3.5 — thinking mode
Reasoning stays **ON** by default.  `ModelBackend` automatically strips `<think>…</think>` blocks from
`output_text` and `gen_token_ids` so verdicts are always clean text.  
The raw thinking content is still accessible via `result.thinking_text`.

### Supported models

| Family | Example IDs | Notes |
|--------|------------|-------|
| **Qwen 3.5** | `Qwen/Qwen3.5-0.8B` · `Qwen/Qwen3.5-4B` · `Qwen/Qwen3.5-9B` · `Qwen/Qwen3.5-27B` · `Qwen/Qwen3.5-35B-A3B` (MoE) · `Qwen/Qwen3.5-122B-A10B` (MoE) | Thinking stripped automatically |
| **Qwen 2.5** | `Qwen/Qwen2.5-{0.5,1.5,3,7,14,32,72}B-Instruct` | Standard path |
| **LLaMA 3.x** | `meta-llama/Llama-3.3-70B-Instruct` | Standard path |
| **Gemma 3/4** | `google/gemma-3-{4b,12b,27b}-it` | Standard path |
| **K2-V2** | `LLM360/K2-V2-Instruct` | Use `K2V2Backend` for `reasoning_effort` |

## 0 — Install

In [ ]:
# Install the package + all ML dependencies from GitHub:
!pip install -q git+https://github.com/YOUR_USERNAME/reagent.git

# Or, if you cloned the repo locally:
# !pip install -q -e ..

# Qwen3.5 is a new architecture — install the latest transformers from source:
# (skip this line for older model families like Qwen2.5, LLaMA, Gemma)
!pip install -q --upgrade "git+https://github.com/huggingface/transformers.git@main"

In [ ]:
import time
import torch
import pandas as pd

from selfie_k2v2 import (
    ModelBackend,
    K2V2Backend,
    HiddenStateProjector,
    make_graph,
    DEFAULT_WRITER_SELFPROBE_SYSTEM,
    DEFAULT_WRITER_SELFPROBE_USER,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

## 1 — Choose your model(s)

Writer and editor can be the **same** instance (shared weights) or
**different** `ModelBackend` instances for cross-model experiments.

### 1a — Qwen 3.5 (all sizes)

Reasoning is **ON**. `<think>…</think>` blocks are stripped automatically from verdicts.  
Access raw thinking via `final["writer_result"].thinking_text`.

| Model ID | Params | VRAM (bf16) | Notes |
|----------|--------|-------------|-------|
| `Qwen/Qwen3.5-0.8B` | 0.9 B | ~2 GB | dev / smoke-test |
| `Qwen/Qwen3.5-4B` | ~4 B | ~8 GB | single consumer GPU |
| `Qwen/Qwen3.5-9B` | ~9 B | ~18 GB | single A100-40 GB |
| `Qwen/Qwen3.5-27B` | 28 B | ~55 GB | 2× A100-40 GB or 1× H100 |
| `Qwen/Qwen3.5-35B-A3B` | 36 B total / ~3 B active | ~18 GB active | MoE — fast |
| `Qwen/Qwen3.5-122B-A10B` | 122 B / ~10 B active | ~40 GB active | MoE — large |
| `Qwen/Qwen3.5-396B-A17B` | 396 B / ~17 B active | ~70 GB active | MoE — flagship |

In [ ]:
# ── Qwen 3.5 ─────────────────────────────────────────────────────────────────
# Reasoning stays ON. <think>…</think> stripped from verdicts automatically.
# Uncomment one size.

# QWEN35_ID = "Qwen/Qwen3.5-4B"           # ← change me
QWEN35_ID = "Qwen/Qwen3.5-0.8B"       # smallest — smoke tests
# QWEN35_ID = "Qwen/Qwen3.5-9B"
# QWEN35_ID = "Qwen/Qwen3.5-27B"
# QWEN35_ID = "Qwen/Qwen3.5-35B-A3B"    # MoE, ~3B active
# QWEN35_ID = "Qwen/Qwen3.5-122B-A10B"  # MoE, ~10B active

t0 = time.time()
backend = ModelBackend(
    model_name=QWEN35_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    # To hard-disable thinking (faster, less compute):
    # chat_template_kwargs={"enable_thinking": False},
)
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"  hidden_size      = {backend.hidden_size}")
print(f"  num_layers       = {backend.num_layers}")
print(f"  strip_thinking   = {backend._strip_thinking}")

writer_backend = editor_backend = backend

In [ ]:
# ── Qwen 3.5 with 4-bit quantization (fits on a 16 GB GPU) ───────────────────
# from transformers import BitsAndBytesConfig
# bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
#                          bnb_4bit_quant_type="nf4")
# backend = ModelBackend("Qwen/Qwen3.5-27B", dtype=torch.bfloat16,
#                        device_map="auto", quantization_config=bnb)
# writer_backend = editor_backend = backend

### 1b — Other supported models

In [ ]:
# ── Other families (uncomment one) ───────────────────────────────────────────

# Qwen 2.5 (sizes: 0.5B / 1.5B / 3B / 7B / 14B / 32B / 72B)
# backend = ModelBackend("Qwen/Qwen2.5-7B-Instruct",
#                        dtype=torch.bfloat16, device_map="auto")

# LLaMA 3.3
# backend = ModelBackend("meta-llama/Llama-3.3-70B-Instruct",
#                        dtype=torch.bfloat16, device_map="auto")

# Gemma 3 (4B / 12B / 27B)
# backend = ModelBackend("google/gemma-3-27b-it",
#                        dtype=torch.bfloat16, device_map="auto")

# K2-V2 — K2V2Backend sets reasoning_effort="medium" automatically
# backend = K2V2Backend(dtype=torch.bfloat16, device_map="auto")

# writer_backend = editor_backend = backend

### 1c — Cross-model (different writer / editor)

In [ ]:
# writer_backend = ModelBackend("Qwen/Qwen3.5-9B",
#                               dtype=torch.bfloat16, device_map="auto")
# editor_backend = ModelBackend("Qwen/Qwen3.5-27B",
#                               dtype=torch.bfloat16, device_map="auto")
# proj = HiddenStateProjector.between(writer_backend, editor_backend)
# print(f"Projection: {proj.src_size} → {proj.tgt_size}  needs={proj.needs_projection}")

## 2 — Sanity check

In [ ]:
t0 = time.time()
ids = writer_backend.build_chat_prompt(
    system="You are a helpful assistant.",
    user="Say hi in 5 words.",
)
# capture_hidden=True so we can verify HS covers ALL answer tokens
r = writer_backend.generate(ids, max_new_tokens=256, capture_hidden=True)
print(f"[{time.time()-t0:.1f}s]")
print(f"output_text   : {r.output_text!r}")
print(f"answer_offset : {r.answer_offset}  (thinking tokens stripped)")
if r.thinking_text:
    print(f"thinking_text : {r.thinking_text[:120]}...")

# ── Verify hidden-state coverage ─────────────────────────────────────────────
# The HS tensor must cover every position in output_ids, so every answer token
# (and its final token) has a captured hidden state for injection / SELFIE.
hs_len   = r.hidden_states[-1].shape[1]
full_len = r.output_ids.shape[1]
ans_start = r.prompt_len + r.answer_offset
ans_end   = ans_start + len(r.gen_token_ids)
print(f"prompt_len    = {r.prompt_len}")
print(f"answer_offset = {r.answer_offset}  (thinking tokens skipped)")
print(f"answer tokens = {len(r.gen_token_ids)}  positions [{ans_start}, {ans_end})")
print(f"HS length     = {hs_len}  full_len = {full_len}")
print(f"HS covers     = {'ALL' if hs_len == full_len else f'{hs_len}/{full_len}'} tokens")

assert r.output_text.strip(), "Empty output — check model load or chat template."
assert "<think>" not in r.output_text, (
    "Got a raw <think> block in output_text — thinking stripping failed. "
    "Check that _strip_thinking is True for Qwen3/3.5 models."
)
assert hs_len == full_len, (
    f"HS length ({hs_len}) < output length ({full_len}). "
    "Call generate(..., capture_all_tokens=True) — this should be the default."
)

## 3 — Build the graph and run all arms

```
writer → probe_editor            (A: raw text → editor)
             → probe_selfie_writer
             → probe_selfie_editor
             → probe_editor_selfhs     (B: editor's own HS re-injected)
             → probe_editor_writerhs   (C: writer's HS injected into editor)
             → probe_writer_selfprobe  (writer's HS → writer, user-defined prompt)
             → END
```

For Qwen3/3.5: the writer thinks privately, then its clean answer is spliced into the editor prompt.

**Hidden-state coverage:** `ModelBackend.generate()` now closes the inline-capture off-by-one automatically — the final generated token's HS is recomputed via a single post-generation forward pass so injection / SELFIE use **all** answer tokens (not N–1 of N).

**Customizable self-probe prompt:** `WRITER_SELFPROBE_SYSTEM` / `WRITER_SELFPROBE_USER` below control what the writer is asked to do with its own re-injected hidden states. The user-string must contain `<<<INJECT_HERE>>>` exactly once — that's where the HS get spliced.

In [ ]:
TASK = "Describe a golden retriever in 3 sentences."
MAX_WRITER_TOKENS = 80    # plenty for 3 sentences; raise to 300+ when thinking is ON
MAX_EDITOR_TOKENS = 48    # one-sentence verdict
INJECT_LAYER      = 0     # 0 = embedding layer; try num_layers//2 for mid-stack

# ── Speed controls ───────────────────────────────────────────────────────────
# Cost breakdown (one generate() call per ✓):
#   writer            ✓  (required)
#   Arm A editor      ✓  (required; also captures HS used by Arm B)
#   Arm B editor      ✓  (if RUN_ARM_B)         — editor's own HS re-injected
#   Arm C editor      ✓  (required)             — writer's HS → editor
#   writer self-probe ✓  (if RUN_SELF_PROBE)    — writer's HS → writer
#   SELFIE writer     ✓ × SELFIE_NUM_LAYERS  (if RUN_SELFIE)
#   SELFIE editor     ✓ × SELFIE_NUM_LAYERS  (if RUN_SELFIE)
#
# Fastest (just the core 3-arm comparison) = 4 generate() calls:
#     RUN_SELFIE=False, RUN_SELF_PROBE=False, RUN_ARM_B=False  (A + C only = 3)
#     RUN_SELFIE=False, RUN_SELF_PROBE=False, RUN_ARM_B=True   (A + B + C  = 4)
# On Qwen 3.5 27B that's ~1-2 min.  Adding SELFIE + self-probe is ~5× slower.
RUN_SELFIE       = False  # turn on only when you want the SELFIE analysis tables
RUN_SELF_PROBE   = True   # sanity check: does writer reconstruct its HS?
RUN_ARM_B        = True   # editor's own HS re-injected comparison
SELFIE_NUM_LAYERS = 2     # 1 = fastest; 2 = default; 3 = thorough
SELFIE_MAX_POSITIONS = 8  # token positions per layer; driven as a single batch
VERBOSE_TIMING   = True   # print wall-clock time per probe node

# ── Writer self-probe prompts ────────────────────────────────────────────────
# The user-prompt MUST contain exactly one "<<<INJECT_HERE>>>" marker — that
# position is where the writer's own final hidden states get spliced into the
# residual stream.  Change these to change what the writer is asked to do with
# its injected HS (defaults ask it to reconstruct/repeat the encoded content).
#
# Examples to try:
#   "The message says: '<<<INJECT_HERE>>>'. In one sentence, what topic does it cover?"
#   "Here is an encoded thought: '<<<INJECT_HERE>>>'. Translate it into plain English:"
WRITER_SELFPROBE_SYSTEM = DEFAULT_WRITER_SELFPROBE_SYSTEM
WRITER_SELFPROBE_USER   = DEFAULT_WRITER_SELFPROBE_USER

graph = make_graph(
    writer_backend,
    editor_backend,
    max_writer_tokens=MAX_WRITER_TOKENS,
    max_editor_tokens=MAX_EDITOR_TOKENS,
    inject_layer=INJECT_LAYER,
    run_selfie=RUN_SELFIE,
    run_self_probe=RUN_SELF_PROBE,
    run_arm_b=RUN_ARM_B,
    selfie_num_layers=SELFIE_NUM_LAYERS,
    selfie_max_positions=SELFIE_MAX_POSITIONS,
    writer_selfprobe_system=WRITER_SELFPROBE_SYSTEM,
    writer_selfprobe_user=WRITER_SELFPROBE_USER,
    verbose_timing=VERBOSE_TIMING,
)

print(f"Task: {TASK!r}")
t0 = time.time()
final = graph.invoke({"task": TASK})
print(f"Done in {time.time()-t0:.1f}s")

## 4 — Three-arm comparison

In [ ]:
print("══ WRITER OUTPUT (clean — thinking stripped) " + "═"*28)
print(final["writer_output_text"])

# Peek at thinking if present
wr = final["writer_result"]
if wr.thinking_text:
    print(f"\n  [thinking: {len(wr.thinking_text)} chars, {wr.answer_offset} thinking tokens]")

print("\n── Arm A  (raw text → editor) " + "─"*42)
print(final["editor_verdict"])

print("\n── Arm B  (editor's own HS re-injected) " + "─"*31)
print(final["editor_selfhs_verdict"])

print("\n── Arm C  (writer's HS injected into editor) " + "─"*27)
print(final["editor_writerhs_verdict"])

print("\n── Writer self-probe  (writer's HS → writer, 'repeat this') " + "─"*10)
print(final["writer_selfprobe_output"])

print()
print("A ≈ B              →  editor's internal repr ≡ surface text")
print("A ≈ C              →  writer's hidden states ≡ writer's text")
print("B ≈ C              →  writer and editor share compatible representations")
print("self-probe ≈ essay →  writer's HS genuinely encode the essay content")

## 5 — (Optional) Inspect writer's thinking

The raw chain-of-thought is preserved in `result.thinking_text` even though it is
stripped from verdicts and token sequences.

In [ ]:
wr = final["writer_result"]
if wr.thinking_text:
    print(f"Writer thinking ({wr.answer_offset} tokens):")
    print(wr.thinking_text)
else:
    print("No thinking block (non-Qwen3 model or thinking was empty)")

## 6 — SELFIE: writer's hidden states

Each row = one `(layer, token)` from the **answer** portion of the writer's output
(thinking tokens are excluded — their positions are skipped via `answer_offset`).

In [ ]:
df_writer = final["selfie_writer"]
print(f"{len(df_writer)} rows  |  probe layers: {sorted(df_writer['layer'].unique())}\n")
df_writer

## 7 — SELFIE: editor's hidden states at the draft span

In [ ]:
df_editor = final["selfie_editor_on_draft"]
print(f"{len(df_editor)} rows  |  probe layers: {sorted(df_editor['layer'].unique())}\n")
df_editor

## 8 — Side-by-side SELFIE alignment

In [ ]:
wr = final["writer_result"]
# answer_start_in_output_ids = prompt_len + answer_offset
writer_first = wr.prompt_len + wr.answer_offset

w = df_writer.copy()
w["rel_pos"] = w["token_idx"] - writer_first
w = w.rename(columns={"token": "writer_tok", "interpretation": "writer_interp"})

e = df_editor.copy()
e["rel_pos"] = e["token_idx"] - final["editor_draft_start"]
e = e.rename(columns={"token": "editor_tok", "interpretation": "editor_interp"})

merged = (
    w.merge(e[["layer", "rel_pos", "editor_tok", "editor_interp"]],
            on=["layer", "rel_pos"], how="inner")
    [["layer", "rel_pos", "writer_tok", "editor_tok", "writer_interp", "editor_interp"]]
    .sort_values(["layer", "rel_pos"])
    .reset_index(drop=True)
)
merged

## 9 — Plug into an existing LangGraph workflow

In [ ]:
from selfie_k2v2 import SelfieProbeMixin, add_probe_to_graph
from langgraph.graph import StateGraph, END

class MyState(SelfieProbeMixin, total=False):
    task: str

def my_writer(state: MyState) -> MyState:
    prompt_ids = writer_backend.build_chat_prompt(
        system="You are a helpful writing assistant.",
        user=state["task"],
    )
    result = writer_backend.generate(prompt_ids, max_new_tokens=256, capture_hidden=True)
    return {
        **state,
        "writer_result": result,
        "writer_output_text": result.output_text,       # clean — thinking stripped
        "writer_output_token_ids": result.gen_token_ids, # answer tokens only
    }

g = StateGraph(MyState)
g.add_node("my_writer", my_writer)
g.set_entry_point("my_writer")
add_probe_to_graph(g, writer_backend, editor_backend, entry_node="my_writer")
app = g.compile()

out = app.invoke({"task": "Explain why the sky is blue in two sentences."})
print("Arm A:", out["editor_verdict"])
print("Arm B:", out["editor_selfhs_verdict"])
print("Arm C:", out["editor_writerhs_verdict"])

## 10 — Save outputs (optional)

In [ ]:
import os

out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)

df_writer.to_csv(f"{out_dir}/selfie_writer.csv", index=False)
df_editor.to_csv(f"{out_dir}/selfie_editor_on_draft.csv", index=False)
merged.to_csv(f"{out_dir}/merged_side_by_side.csv", index=False)

wr = final["writer_result"]
with open(f"{out_dir}/arm_verdicts.txt", "w") as f:
    f.write(f"model        : {writer_backend.model_name}\n")
    f.write(f"task         : {TASK}\n")
    f.write(f"thinking_len : {wr.answer_offset} tokens\n\n")
    if wr.thinking_text:
        f.write(f"=== Writer thinking ===\n{wr.thinking_text}\n\n")
    f.write(f"=== Writer output (clean) ===\n{final['writer_output_text']}\n\n")
    f.write(f"=== Arm A: editor_verdict (raw text) ===\n{final['editor_verdict']}\n\n")
    f.write(f"=== Arm B: editor_selfhs_verdict ===\n{final['editor_selfhs_verdict']}\n\n")
    f.write(f"=== Arm C: editor_writerhs_verdict ===\n{final['editor_writerhs_verdict']}\n\n")
    f.write(f"=== Writer self-probe (HS → writer, repeat) ===\n{final['writer_selfprobe_output']}\n")

print(f"Saved to ./{out_dir}/")